Checklist

 - Add GNN layer (currently 2, need 3)
 - Reduce dimension of GNN layer
 - Add relevant features in featurizer
 - Check dimensions at each layer
 - Hyperparameter tuning

In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import MolsToGridImage

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

np.random.seed(42)

In [23]:
import torch
from torch.nn import Linear
import torch.nn.functional as F 
from torch_geometric.nn import GCNConv, TopKPooling
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
embedding_size = 64

In [24]:
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import BondType

class Featurizer:
    def __init__(self, allowable_sets, direct_features=None):
        """
        Base featurizer class. 
        - `allowable_sets`: Dict of categorical features (one-hot encoded).
        - `direct_features`: Dict of numerical features (used as direct values).
        """
        self.dim = 0
        self.features_mapping = {}
        self.direct_features = direct_features if direct_features else {}

        # One-hot encoded categorical features
        for k, s in allowable_sets.items():
            s = sorted(list(s))
            self.features_mapping[k] = dict(zip(s, range(self.dim, len(s) + self.dim)))
            self.dim += len(s)

        # Direct numerical features (not one-hot)
        for k in self.direct_features.keys():
            self.features_mapping[k] = self.dim
            self.dim += 1

    def encode(self, inputs):
        output = np.zeros((self.dim,))

        # One-hot encoded features
        for name_feature, feature_mapping in self.features_mapping.items():
            if name_feature in self.direct_features:  # Skip direct features in one-hot encoding
                continue
            feature = getattr(self, name_feature)(inputs)
            if feature not in feature_mapping:
                continue
            output[feature_mapping[feature]] = 1.0

        # Direct numerical features
        for name_feature, index in self.direct_features.items():
            feature = getattr(self, name_feature)(inputs)
            output[index] = feature  # Assign directly

        return output


class AtomFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def symbol(self, atom):
        return atom.GetSymbol()

    def n_hydrogens(self, atom):
        return atom.GetTotalNumHs() if atom.HasProp("_TotalNumHs") else 0

    def hybridization(self, atom):
        return atom.GetHybridization().name.lower()

    def is_aromatic(self, atom):
        return atom.GetIsAromatic()

    def n_valence(self, atom):
        return atom.GetTotalValence()

    def formal_charge(self, atom):
        return atom.GetFormalCharge()  # Direct feature, not one-hot

    def atomic_number(self, atom):
        return atom.GetAtomicNum()  # Direct feature


class BondFeaturizer(Featurizer):
    def __init__(self, allowable_sets, direct_features):
        super().__init__(allowable_sets, direct_features)

    def bond_type(self, bond):
        return bond.GetBondType().name.lower()

    def conjugated(self, bond):
        return bond.GetIsConjugated()

    def bond_order(self, bond):
        """Returns a numeric bond order instead of a categorical one-hot encoding."""
        bond_orders = {
            BondType.SINGLE: 1.0,
            BondType.DOUBLE: 2.0,
            BondType.TRIPLE: 3.0,
            BondType.AROMATIC: 1.5,
        }
        return bond_orders.get(bond.GetBondType(), 0.0)


# Instantiate with updated feature sets
atom_featurizer = AtomFeaturizer(
    allowable_sets={
        "symbol": {"B", "Br", "C", "Ca", "Cl", "F","Ga", "H", "I", "N", "Na", "O", "P", "S","Sb","Se", "Mo", "Nb"},
        "n_hydrogens": {0, 1, 2, 3, 4},
        "hybridization": {"s", "sp", "sp2", "sp3"},
    },
    direct_features={
        "formal_charge": None,  # This will be assigned a direct index in `Featurizer`
        "atomic_number": None,
        # "n_valence": None,  # Direct numeric feature
        "is_aromatic": None,  # Direct boolean feature
        "n_hydrogens": None,  # Direct numeric feature
    }
)

bond_featurizer = BondFeaturizer(
    allowable_sets={
        "bond_type": {"single", "double", "triple", "aromatic"},
        "conjugated": {True, False},
    },
    direct_features={
        "bond_order": None,  # Direct numeric feature
    }
)

# Get the new feature dimensions
node_dim = atom_featurizer.dim
edge_dim = bond_featurizer.dim

print(f"Node feature dimension: {node_dim}")
print(f"Edge feature dimension: {edge_dim}")


Node feature dimension: 31
Edge feature dimension: 7


In [25]:
import torch
import numpy as np
from rdkit import Chem
from torch_geometric.data import Data

# Ensure you define atom_featurizer and bond_featurizer before using them

def molecule_from_smiles(smiles):
    """Convert SMILES to RDKit molecule with error handling."""
    molecule = Chem.MolFromSmiles(smiles, sanitize=False)
    flag = Chem.SanitizeMol(molecule, catchErrors=True)
    if flag != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(molecule, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ flag)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    return molecule

def graph_from_molecule(molecule):
    """Convert RDKit molecule to PyTorch Geometric graph representation."""
    atom_features = []
    bond_features = []
    edge_index = []
    edge_attr = []

    # Chem.SanitizeMol(molecule)  # Ensure the molecule is sanitized

    for atom in molecule.GetAtoms():
        atom_features.append(atom_featurizer.encode(atom))  # Encode atom features

    for bond in molecule.GetBonds():
        start, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([start, end])
        edge_index.append([end, start])  # Ensure undirected edges
        bond_features.append(bond_featurizer.encode(bond))  # Bond features
        bond_features.append(bond_featurizer.encode(bond))  # Reverse edge

    # Convert to PyTorch tensors
    x = torch.tensor(atom_features, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bond_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def graphs_from_smiles(smiles_list):
    """Convert a list of SMILES strings to PyTorch Geometric Data objects."""
    graphs = []
    for smiles in smiles_list:
        molecule = molecule_from_smiles(smiles)
        graph = graph_from_molecule(molecule)
        graphs.append(graph)
    return graphs  # This can be used with a PyG DataLoader


In [26]:
from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from pymatgen.analysis.local_env import CutOffDictNN
from scipy.spatial import distance_matrix
from rdkit import Chem
from rdkit.Chem import AllChem, Draw

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def cif_to_mol(cif_file):
    structure = Structure.from_file(cif_file)

    mol = Chem.RWMol()  # Create an editable molecule in RDKit

    for site in structure:
        element = site.specie.symbol  # Get atomic symbol (e.g., "C", "O")
        atom = Chem.Atom(element)  # Create an RDKit atom
        mol.AddAtom(atom)  # Add it to the molecule

    #print(structure[0].coords)

    cutoff = 3 # Example: typical bond length for C-C or C-H

    for i in range(len(structure)):
        for j in range(i + 1, len(structure)):
            dist = structure.get_distance(i, j)
            if dist < cutoff:  # Only consider distances within the cutoff
                #print(f"Bond between atom {i} and atom {j}: {dist:.3f} Å")
                if mol.GetBondBetweenAtoms(i, j) is None:  
                    mol.AddBond(i, j, Chem.BondType.SINGLE)  # Only add if not present

    conf = Chem.Conformer(mol.GetNumAtoms())  # Create a conformer

    for idx, site in enumerate(structure):
        coord = site.coords  # Cartesian coordinates (x, y, z)
        conf.SetAtomPosition(idx, coord)  # Assign position

    mol.AddConformer(conf)  # Add conformer to molecule

    return mol

In [54]:
from torch_geometric.data import Data

class PairedData(Data):
    def __init__(self, data1, data2, y, weight=1.0):
        super().__init__()
        self.x1 = data1.x
        self.edge_index1 = data1.edge_index
        self.edge_attr1 = data1.edge_attr
        
        self.x2 = data2.x
        self.edge_index2 = data2.edge_index
        self.edge_attr2 = data2.edge_attr
        
        self.y = float(str(y).replace("−", "+"))  # Target value for the pair
        self.w = torch.tensor([weight], dtype=torch.float)

    def __inc__(self, key, value, *args, **kwargs):
        """Ensures proper indexing when batching."""
        if key == "edge_index1":
            return self.x1.shape[0] if self.x1 is not None else 0
        if key == "edge_index2":
            return self.x2.shape[0] if self.x2 is not None else 0
        return super().__inc__(key, value, *args, **kwargs)
    


In [56]:
import pandas as pd
import os

df = pd.read_csv(r"trial_dataCopy.csv") #TODO

file_path_cif = r"CIF_files" 
file_path_sdf = r"SDF_files" 

materials = []
drugs = []

for cif in df["material"]:
    file = os.path.join(file_path_cif, cif + ".cif")
    print(f"Processing CIF: {file}")  # Progress tracking
    mol = cif_to_mol(file)
    materials.append(graph_from_molecule(mol))
print("Finished processing all CIF files.\n")

for sdf in df["drug"]:
    file = os.path.join(file_path_sdf, sdf + ".sdf")
    print(f"Processing SDF: {file}")  # Progress tracking
    supplier = Chem.SDMolSupplier(file)
    mol = supplier[0]
    if mol is None:
        print(f"Warning: Failed to read molecule from {file}")  # Handle errors
    drugs.append(graph_from_molecule(mol))
print("Finished processing all SDF files.\n")

from torch_geometric.utils import add_self_loops

def add_loops_to_data(data):
    edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.x.size(0))
    data.edge_index = edge_index
    return data

print("Pairing materials and drugs...")
paired_data_list = []
for i, (mat, drug, target, weight) in enumerate(zip(materials, drugs, df["y"], df["weight"])):
    print(f"Pairing {i+1}/{len(df)}: Material-{i}, Drug-{i}, Target-{target}")
    print(f"Material graph: {mat}, Drug graph: {drug}")  # Debugging output

    mat = add_loops_to_data(mat)
    drug = add_loops_to_data(drug)

    paired_data_list.append(PairedData(mat, drug, target, weight))
    #paired_data_list.append(PairedData(drug, mat, target))


print("Finished pairing all data.")


Processing CIF: CIF_files\BC3.cif
Processing CIF: CIF_files\BC3.cif
Processing CIF: CIF_files\Biphenylene.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BN.cif
Processing CIF: CIF_files\BNNT.cif
Processing CIF: CIF_files\BNNT.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\graphene.cif
Processing CIF: CIF_files\Graphyne.cif
Processing CIF: CIF_files\Graphyne.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\Twin-Gr.cif
Processing CIF: CIF_files\molybdenum_disulfide.cif
Processing CIF: CIF_files\molybdenum_disulfide.cif
Processing CIF: CIF_files\phosphorene.cif
Processing CIF

In [57]:
from sklearn.model_selection import train_test_split

# Define train-test split ratio (e.g., 80% train, 20% test)
train_data, test_data = train_test_split(paired_data_list, test_size=0.33, random_state=4)

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")

print(train_data)

Training set size: 30
Testing set size: 15
[PairedData(x1=[4, 31], edge_index1=[2, 12], edge_attr1=[8, 7], x2=[30, 31], edge_index2=[2, 96], edge_attr2=[66, 7], y=0.59, w=[1]), PairedData(x1=[80, 31], edge_index1=[2, 736], edge_attr1=[656, 7], x2=[10, 31], edge_index2=[2, 32], edge_attr2=[22, 7], y=1.4557, w=[1]), PairedData(x1=[2, 31], edge_index1=[2, 4], edge_attr1=[2, 7], x2=[12, 31], edge_index2=[2, 34], edge_attr2=[22, 7], y=4.171, w=[1]), PairedData(x1=[80, 31], edge_index1=[2, 880], edge_attr1=[800, 7], x2=[17, 31], edge_index2=[2, 53], edge_attr2=[36, 7], y=0.66, w=[1]), PairedData(x1=[18, 31], edge_index1=[2, 228], edge_attr1=[210, 7], x2=[9, 31], edge_index2=[2, 27], edge_attr2=[18, 7], y=0.43, w=[1]), PairedData(x1=[4, 31], edge_index1=[2, 12], edge_attr1=[8, 7], x2=[25, 31], edge_index2=[2, 77], edge_attr2=[52, 7], y=2.12, w=[1]), PairedData(x1=[8, 31], edge_index1=[2, 32], edge_attr1=[24, 7], x2=[12, 31], edge_index2=[2, 34], edge_attr2=[22, 7], y=1.18, w=[1]), PairedData(

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader  # Assuming PyG DataLoader

# Define training function
def train(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)  # Move batch to GPU if available
        
        optimizer.zero_grad()  # Reset gradients
        output = model(batch)  # Forward pass

        tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

        loss = torch.sqrt(criterion(output, tensor_x))  # RMSE Loss
        weighted_loss = (loss * batch.w)
        weighted_loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()
    
    return total_loss / len(train_loader)  # Return average loss

# Define evaluation function
def evaluate(model, val_loader, criterion, device):

    model.eval()  # Set model to evaluation mode
    total_loss = 0

    with torch.no_grad():  # Disable gradient tracking
        for batch in val_loader:
            batch = batch.to(device)
        
            output = model(batch)

            tensor_x = torch.tensor([batch.y], dtype=torch.float).to(device)  # Convert target to tensor 

            loss = torch.sqrt(criterion(output,tensor_x)) # RMSE Loss
            total_loss += np.absolute(loss.item())

    return total_loss / len(val_loader)
    


In [47]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp


# === Message-passing layer with edge features ===
class MPNN(MessagePassing):
    def __init__(self, in_dim, edge_dim, out_dim, hidden_dim=32, aggr="max"):
        super().__init__(aggr=aggr)
        self.mlp = nn.Sequential(
            nn.Linear(in_dim + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        msg_input = torch.cat([x_j, edge_attr], dim=-1)
        return self.mlp(msg_input)

    def update(self, aggr_out):
        return aggr_out


# === Graph encoder with 3 layers and pooling ===
class MPNNEncoder(nn.Module):
    def __init__(self, node_in_dim, edge_dim, hidden_dim, out_dim, num_layers=5, dropout=0.2):
        super().__init__()
        layers = []
        in_dim = node_in_dim
        for i in range(num_layers - 1):
            layers.append(MPNN(in_dim, edge_dim, hidden_dim))
            in_dim = hidden_dim
        layers.append(MPNN(in_dim, edge_dim, out_dim))
        self.layers = nn.ModuleList(layers)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr, batch=None):
        for i, conv in enumerate(self.layers):
            x = conv(x, edge_index, edge_attr)
            if i != len(self.layers) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        # Graph-level embedding (max ⊕ mean pooling)
        if batch is None:
            pooled = torch.cat([
                x.max(dim=0, keepdim=True)[0],
                x.mean(dim=0, keepdim=True)
            ], dim=1)     # → [1, 2*out_dim]
        else:
            pooled = torch.cat([gmp(x, batch), gap(x, batch)], dim=1)  # [B, 2*out_dim]
        return pooled


# === Regressor combining two graphs ===
class MPNNRegressor(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=64, gnn_out_dim=128, mlp_hidden=64):
        super().__init__()
        self.encoder1 = MPNNEncoder(node_dim, edge_dim, hidden_dim, gnn_out_dim)
        self.encoder2 = MPNNEncoder(node_dim, edge_dim, hidden_dim, gnn_out_dim)

        self.mlp = nn.Sequential(
            nn.Linear(4 * gnn_out_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, data):
        h1 = self.encoder1(data.x1, data.edge_index1, data.edge_attr1, getattr(data, "batch1", None))
        h2 = self.encoder2(data.x2, data.edge_index2, data.edge_attr2, getattr(data, "batch2", None))
        h = torch.cat([h1, h2], dim=-1)  # [B, 4*gnn_out_dim]
        out = self.mlp(h)
        return out.view(-1)


In [33]:
# === GCN encoder (ignores edge features) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)  # <--- edge_attr not used
            if i != len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        # return graph-level embedding by mean-pooling over nodes
        return torch.cat([gap(x,batch=None), gmp(x,batch=None)], dim=1)   # single-graph pooling


# === Regressor that combines two graphs ===
class GCNRegressor(nn.Module):
    def __init__(self, node_dim, hidden_dim=64, gnn_out_dim=128, mlp_hidden=64):
        super().__init__()
        self.encoder1 = GCNEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.encoder2 = GCNEncoder(node_dim, hidden_dim, gnn_out_dim)
        
        self.mlp = nn.Sequential(
            nn.Linear(4 * gnn_out_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1)   # regression output
        )

    def forward(self, data):
        h1 = self.encoder1(data.x1, data.edge_index1)
        h2 = self.encoder2(data.x2, data.edge_index2)
        h = torch.cat([h1, h2], dim=-1)
        out = self.mlp(h)
        return out.view(-1)


In [34]:
from torch_geometric.nn import GATConv

class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, num_layers=2, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout))
        
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout))
        
        self.convs.append(GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout))
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i != len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class GATRegressor(nn.Module):
    def __init__(self, node_dim, hidden_dim=64, gnn_out_dim=64):
        super().__init__()
        self.encoder1 = GATEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.encoder2 = GATEncoder(node_dim, hidden_dim, gnn_out_dim)
        self.fc = nn.Sequential(
            nn.Linear(4 * gnn_out_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, data):
        # First graph (material)
        x1 = self.encoder1(data.x1, data.edge_index1)
        h1 = torch.cat([gmp(x1,batch=None), gap(x1, batch=None)], dim=1)


        # Second graph (drug)
        x2 = self.encoder2(data.x2, data.edge_index2)
        h2 = torch.cat([gmp(x2,batch=None), gap(x2, batch=None)], dim=1)


        # Concatenate + predict
        h = torch.cat([h1, h2], dim=-1)
        return self.fc(h).squeeze()


In [39]:
# === Transformer over concatenated graphs (pairwise regression) ===
import torch
import torch.nn as nn
import torch.nn.functional as F

def build_allowed_mask(n1: int, n2: int, edge_index1: torch.Tensor, edge_index2: torch.Tensor, device=None):
    """
    Returns an [L, L] mask of allowed attentions (1=allow, 0=block),
    with adjacency (incl. self-loops) on the diagonal blocks, and all-ones off-diagonal.
    """
    L = n1 + n2
    allowed = torch.zeros((L, L), dtype=torch.bool, device=device)

    # --- material block A1 ---
    A1 = torch.zeros((n1, n1), dtype=torch.bool, device=device)
    if edge_index1.numel() > 0:
        src, dst = edge_index1  # edges j->i
        A1[dst, src] = True
        A1[src, dst] = True   # symmetrize; safe for undirected use
    A1.fill_diagonal_(True)   # self-loops
    allowed[:n1, :n1] = A1

    # --- drug block A2 ---
    A2 = torch.zeros((n2, n2), dtype=torch.bool, device=device)
    if edge_index2.numel() > 0:
        src2, dst2 = edge_index2
        A2[dst2, src2] = True
        A2[src2, dst2] = True
    A2.fill_diagonal_(True)
    allowed[n1:, n1:] = A2

    # --- off-diagonals: full cross attention ---
    allowed[:n1, n1:] = True
    allowed[n1:, :n1] = True

    return allowed  # [L, L] bool

class TransformerRegressor(nn.Module):
    """
    - Projects node features to d_model
    - Adds token-type embedding (0=material, 1=drug)
    - Runs Transformer encoder with a custom attention mask
    - Pools per graph and regresses a scalar
    """
    def __init__(self, node_dim: int, d_model: int = 128, nhead: int = 8,
                 num_layers: int = 4, dim_feedforward: int = 256, dropout: float = 0.1):
        super().__init__()
        self.in_proj = nn.Linear(node_dim, d_model)
        self.type_embed = nn.Embedding(2, d_model)  # 0: material, 1: drug

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(4 * d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1)
        )

    def forward(self, data):
        """
        data: PairedData with fields x1, edge_index1, x2, edge_index2, y (float)
        """
        device = next(self.parameters()).device
        x1 = data.x1.to(device).float()
        x2 = data.x2.to(device).float()
        ei1 = data.edge_index1.to(device).long() if data.edge_index1 is not None else torch.empty((2,0), dtype=torch.long, device=device)
        ei2 = data.edge_index2.to(device).long() if data.edge_index2 is not None else torch.empty((2,0), dtype=torch.long, device=device)

        n1, n2 = x1.size(0), x2.size(0)
        L = n1 + n2

        # Node embeddings + token type embeddings
        h1 = self.in_proj(x1) + self.type_embed(torch.zeros(n1, dtype=torch.long, device=device))   # material
        h2 = self.in_proj(x2) + self.type_embed(torch.ones(n2, dtype=torch.long, device=device))    # drug
        H = torch.cat([h1, h2], dim=0).unsqueeze(0)  # [1, L, d_model] batch_first

        # Build allowed-attention mask and convert to additive attn_mask
        allowed = build_allowed_mask(n1, n2, ei1, ei2, device=device)  # [L, L] bool
        # PyTorch expects attn_mask where True=mask OR a float with -inf where masked.
        # We'll use float additive mask: 0 for allowed, -inf for blocked.
        attn_mask = (~allowed).float() * -1e9  # [L, L]

        # Encoder
        Z = self.encoder(H, mask=attn_mask)  # [1, L, d_model]
        Z_nodes = Z[0]  # [L, d_model]

        z1_mean = Z_nodes[:n1].mean(dim=0, keepdim=True)
        z1_max  = Z_nodes[:n1].max(dim=0, keepdim=True).values
        z1 = torch.cat([z1_mean, z1_max], dim=-1)  # [1, 2*d_model]

        z2_mean = Z_nodes[n1:].mean(dim=0, keepdim=True)
        z2_max  = Z_nodes[n1:].max(dim=0, keepdim=True).values
        z2 = torch.cat([z2_mean, z2_max], dim=-1)  # [1, 2*d_model]

        h = torch.cat([z1, z2], dim=-1)  # [1, 4*d_model]

        out = self.head(h)

        return out




In [60]:
from sklearn.model_selection import KFold
from torch.utils.data import Subset

k = 5
num_epochs = 80
kfold = KFold(n_splits=k, shuffle=True, random_state=25)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_losses = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(paired_data_list)):
    print(f"\nFold {fold+1}/{k}")

    # Subset works fine because paired_data_list is indexable
    train_loader = Subset(paired_data_list, train_idx)
    val_loader = Subset(paired_data_list, val_idx)

    print(f"Training set size: {len(train_loader)}")
    print(f"Validation set size: {len(val_loader)}")

    # Fresh GCN model for each fold
    # model = GCNRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
    # model = GATRegressor(node_dim=node_dim, hidden_dim=64, gnn_out_dim=128).to(device)
    # model = MPNNRegressor(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, gnn_out_dim=128).to(device)
    model = TransformerRegressor(node_dim=node_dim, d_model=128, nhead=8, num_layers=4, dim_feedforward=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-5)
    criterion = torch.nn.MSELoss()

    for epoch in range(0, num_epochs + 1):
        train_loss = train(model, train_loader, optimizer, criterion, device)

        if epoch % 10 == 0:
            val_loss = evaluate(model, val_loader, criterion, device)
            print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

    val_loss = evaluate(model, val_loader, criterion, device)
    val_losses.append(val_loss)

# Final result
mean_val_loss = sum(val_losses) / len(val_losses)
print(f"\n✅ K-Fold Cross-Validation Complete! Mean Validation Loss = {mean_val_loss:.4f}")


Fold 1/5
Training set size: 36
Validation set size: 9
Epoch 0: Train Loss = 2.2348, Val Loss = 0.9351
Epoch 10: Train Loss = 1.1779, Val Loss = 0.2546
Epoch 20: Train Loss = 1.1147, Val Loss = 0.2618
Epoch 30: Train Loss = 1.1300, Val Loss = 0.2643
Epoch 40: Train Loss = 1.1112, Val Loss = 0.2725
Epoch 50: Train Loss = 1.1136, Val Loss = 0.2537
Epoch 60: Train Loss = 1.1350, Val Loss = 0.3212
Epoch 70: Train Loss = 1.1049, Val Loss = 0.3906
Epoch 80: Train Loss = 1.0877, Val Loss = 0.2676

Fold 2/5
Training set size: 36
Validation set size: 9
Epoch 0: Train Loss = 1.7694, Val Loss = 0.7582
Epoch 10: Train Loss = 1.1551, Val Loss = 0.9017
Epoch 20: Train Loss = 1.0587, Val Loss = 0.8488
Epoch 30: Train Loss = 0.9939, Val Loss = 0.6827
Epoch 40: Train Loss = 0.9999, Val Loss = 0.6854
Epoch 50: Train Loss = 0.9702, Val Loss = 0.6797
Epoch 60: Train Loss = 1.0081, Val Loss = 0.6806
Epoch 70: Train Loss = 0.9723, Val Loss = 0.6856
Epoch 80: Train Loss = 0.9914, Val Loss = 0.6880

Fold 3/5
